# UnibaBot PDA -- Fine-tuning Llama 3.2 3B con QLoRA

**Objetivo:** Especializar Llama 3.2 3B en verificacion de cumplimiento de PDAs.

**Requisitos:** Google Colab con GPU T4 (gratuito) o A100 (Pro)

**Dataset:** 42 ejemplos de entrenamiento + 5 de validacion en formato Alpaca

## 1. Instalar dependencias

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps unsloth

## 2. Subir dataset

Sube `training_dataset.jsonl` y `validation_dataset.jsonl` desde `data/` a Colab.

Puedes usar el panel de archivos de Colab (icono de carpeta a la izquierda) o ejecutar la celda siguiente para subirlos.

In [ ]:
from google.colab import files
print("Sube training_dataset.jsonl:")
uploaded = files.upload()
print("Sube validation_dataset.jsonl:")
uploaded = files.upload()

## 3. Cargar modelo con cuantizacion 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None  # auto-detecta: float16 para T4, bfloat16 para A100
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Modelo cargado. Parametros: {model.num_parameters():,}")

## 4. Configurar adaptadores LoRA

LoRA agrega matrices pequenas entrenables a capas especificas del Transformer.
Solo se entrenan ~1-3% de los parametros totales.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # rango de matrices LoRA
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # atencion
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # ahorra 30% VRAM
    random_state=42,
)

# Mostrar parametros entrenables vs totales
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parametros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 5. Preparar dataset

In [ ]:
import json
from datasets import Dataset

def cargar_jsonl(path):
    datos = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            datos.append(json.loads(line))
    return datos

train_data = cargar_jsonl("training_dataset.jsonl")
val_data = cargar_jsonl("validation_dataset.jsonl")

print(f"Train: {len(train_data)} ejemplos")
print(f"Validation: {len(val_data)} ejemplos")

In [ ]:
# Template de prompt para Llama 3.2 Instruct
alpaca_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un evaluador academico de la Universidad de Ibague que verifica el cumplimiento de Planes de Desarrollo Academico (PDA).<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}

{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

def formatear_ejemplo(ejemplo):
    return alpaca_template.format(
        instruction=ejemplo["instruction"],
        input=ejemplo["input"],
        output=ejemplo["output"],
    )

# Convertir a Dataset de HuggingFace
train_dataset = Dataset.from_list(
    [{"text": formatear_ejemplo(e)} for e in train_data]
)
val_dataset = Dataset.from_list(
    [{"text": formatear_ejemplo(e)} for e in val_data]
)

# Verificar un ejemplo
print(train_dataset[0]["text"][:500])
print("...")

## 6. Entrenar con SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # True puede mejorar velocidad pero complica con datasets pequenos
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # batch efectivo = 2 * 4 = 8
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# Verificar uso de memoria antes de entrenar
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memoria reservada: {start_gpu_memory} GB / {max_memory} GB")

In [ ]:
# ENTRENAR
trainer_stats = trainer.train()

print(f"\nEntrenamiento completado en {trainer_stats.metrics['train_runtime']:.0f} segundos")
print(f"Loss final: {trainer_stats.metrics['train_loss']:.4f}")

## 7. Verificar loss curve

El loss debe bajar de forma estable. Si sube o es muy erratico, hay problemas.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_losses = [l["loss"] for l in logs if "loss" in l]
eval_losses = [l["eval_loss"] for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(eval_losses, marker="o")
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

print(f"Train loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}")
print(f"Eval loss por epoch: {[f'{l:.4f}' for l in eval_losses]}")

## 8. Probar el modelo fine-tuneado

In [ ]:
# Probar con un ejemplo del set de validacion
FastLanguageModel.for_inference(model)

test_example = val_data[0]
test_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un evaluador academico de la Universidad de Ibague que verifica el cumplimiento de Planes de Desarrollo Academico (PDA).<|eot_id|><|start_header_id|>user<|end_header_id|>

{test_example['instruction']}

{test_example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,
    use_cache=True,
)

respuesta = tokenizer.batch_decode(outputs)[0]
# Extraer solo la respuesta del asistente
respuesta = respuesta.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
respuesta = respuesta.replace("<|eot_id|>", "").strip()

print("=== RESPUESTA DEL MODELO FINE-TUNEADO ===")
print(respuesta[:1000])

## 9. Guardar adaptador LoRA

In [ ]:
# Guardar solo el adaptador LoRA (no el modelo completo)
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Comprimir para descargar
!zip -r lora_adapter.zip lora_adapter/

print("Adaptador guardado. Tamano:")
!du -sh lora_adapter/
!ls -lh lora_adapter.zip

In [ ]:
# Descargar el adaptador
from google.colab import files
files.download("lora_adapter.zip")

## 10. (Opcional) Exportar a GGUF para ollama

Esto permite correr el modelo fine-tuneado directamente en ollama en tu Mac.

In [ ]:
# Exportar a GGUF Q4_K_M (buena calidad, ~2GB)
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
    quantization_method="q4_k_m",
)

print("Modelo GGUF exportado:")
!ls -lh gguf_model/

In [ ]:
# Descargar el GGUF
import glob
gguf_file = glob.glob("gguf_model/*.gguf")[0]
print(f"Descargando: {gguf_file}")
files.download(gguf_file)

## Instrucciones post-descarga

### Opcion A: Usar el GGUF con ollama (recomendado)

1. Descargar el archivo `.gguf` a tu Mac
2. Crear un Modelfile:
```
FROM ./modelo.gguf
PARAMETER temperature 0.1
SYSTEM Eres un evaluador academico de la Universidad de Ibague que verifica el cumplimiento de Planes de Desarrollo Academico (PDA).
```
3. Registrar en ollama:
```bash
ollama create unibabot-pda -f Modelfile
ollama run unibabot-pda
```

### Opcion B: Usar el adaptador LoRA con Python

1. Descomprimir `lora_adapter.zip` en `models/lora_adapter/`
2. Cargar con transformers + peft en el script del agente